# ML Topics

## 1. Softmax

Think of Softmax as the bridge between raw, unnormalized model outputs (logits) and a valid probability distribution.

The Equation

The Softmax function σ for a vector z of K real numbers is defined as:

$$\sigma(z)_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

* `zi`​: The individual "logit" (the raw score) for class i.

* `e^zi`​: The exponential function applied to the logit. This ensures every value becomes positive.

* `∑ezj​`: The sum of all exponentiated logits. This is the normalization factor.


The output of a Softmax operation has two mathematical properties that are essential for classification:
* Each output is between 0 and 1.
* The sum of all outputs is exactly 1.

Core Principles
1. The "Winner-Take-All" Nature
* Because ex grows exponentially, Softmax doesn't just normalize; it exaggerates differences. A logit that is slightly larger than the others will "steal" most of the probability mass.
    - `Example`: If your logits are [1,2,5], the value 5 is only 2.5× larger than 2, but e5 (148.4) is roughly 20× larger than e2 (7.4).

2. Softmax vs. Temperature (T)
* Sometimes you’ll see the equation with a temperature parameter: ∑ezj​/Tezi​/T​.
    - `High T (>1)`: Makes the distribution "flatter" or more uniform (used in LLM creativity).
    - `Low T (<1)`: Makes the distribution "sharper," pushing it toward a one-hot vector (used for confident, deterministic tasks).

3. Numerical Stability (The "Log-Sum-Exp" Trick)
* In a coding interview, if you implement Softmax manually, it might crash (overflow) if your logits are large (e.g., zi​=1000).
    - To fix this, we subtract the maximum value from all logits before exponentiating: `σ(z)i​=∑ezj​−max(z)ezi​−max(z)​`
    - (Details below)
    - This is numerically stable because the largest value becomes e0=1, and everything else is ≤1.

### a. Softmax Exploding/Vanishing Gradient Problem

**Main Point**: the `log softmax` is used in practice (as opposed to the standard softmax) during model training because it stabilizes exploding/vanishing gradients due to the exponentials (e^z) inside the softmax equation. Standard softmax is used at inference/evaluation for human readable performance / understanding

1. The Exploding Gradient Risk
    * If you have a logit of 1000, e1000 results in inf (Infinity). When your code tries to calculate the gradient of inf, the gradient becomes NaN or a massive number that "explodes" your weights during the update. This is why we use the Log-Sum-Exp trick to stay in the "Log" realm where numbers are smaller and manageable.

2. The Vanishing Gradient Risk (More Common)
    * Standard Softmax is even more notorious for Vanishing Gradients.

    * The Plateau: If a logit is already very high (model is confident), the Softmax curve becomes very flat.

    * The Math: The derivative of Softmax involves σ(1−σ). If the output σ is near 1.0 (or near 0.0), the derivative becomes nearly zero.

    * The Result: During backprop, you multiply the loss by this near-zero derivative. The update to your early layers becomes non-existent, and the model stops learning. Log-Softmax fixes this because the derivative of a log-exponential is much more linear and doesn't "flatline" as easily.


If the interviewer asks: `"Why are my gradients NaN?"`

* Check 1: Are you using raw Softmax in the loss function? (Switch to Log-Softmax).

* Check 2: Are the input features normalized? (Large inputs lead to large logits, which lead to inf exponentials).

* Check 3: Is the learning rate too high? (Aggressive steps can push weights into the "exploding" exponential territory).

The standard Softmax for a single class xi​ is:

$$\sigma(x_i) = \frac{e^{x_i}}{\sum e^{x_j}}$$

When we take the Log of that entire fraction, the rules of logarithms `(log(a/b)=loga−logb)` allow us to break it apart:

$$\log(\sigma(x_i)) = \log(e^{x_i}) - \log\left(\sum_{j} e^{x_j}\right)$$

Since `log(e^xi​)` is just `xi`​, the equation becomes:

$$\text{Log-Softmax} = x_i - \underbrace{\log\left(\sum_{j=1}^{n} e^{x_j}\right)}_{\text{Log-Sum-Exp}}$$

### b. BUT this still doesn't solve the exploding gradient issue because we're still calculation `e^x`

We add a mathematical `trick` of a `Mathematical Zero` of `zmax` to stabilize the finite-bit computation calculation that is practical for most modern computers calculating 16-bit, 32-bit, 64-bit floats.

Think of it like the difference between `10−(2+3)` and `(10−2)−3`. Both equal `5`.

1. Definition:
$$\text{LogSoftmax}(x_i) = x_i - \log \sum e^{x_j}$$

2. Injecting the Shift (The Bridge):
$$\text{LogSoftmax}(x_i) = x_i - \log \sum e^{x_j - z_{\text{max}} + z_{\text{max}}}$$

$$\sum e^{x_j - z_{\text{max}} + z_{\text{max}}} = \sum \left( e^{x_j - z_{\text{max}}} \cdot e^{z_{\text{max}}} \right)$$

$$\sum \left( e^{x_j - z_{\text{max}}} \cdot e^{z_{\text{max}}} \right) = e^{z_{\text{max}}} \cdot \left( \sum e^{x_j - z_{\text{max}}} \right)$$

$$\log \left( e^{z_{\text{max}}} \cdot \sum e^{x_j - z_{\text{max}}} \right) = \log(e^{z_{\text{max}}}) + \log \left( \sum e^{x_j - z_{\text{max}}} \right)$$

$$z_{\text{max}} + \log \sum e^{x_j - z_{\text{max}}}$$

3. Substituting the Stable Identity:
$$\text{LogSoftmax}(x_i) = x_i - \left(z_{\text{max}} + \log \sum e^{x_j - z_{\text{max}}}\right)$$

4. Distributing the Negative:
$$\text{LogSoftmax}(x_i) = x_i - z_{\text{max}} - \log \sum e^{x_j - z_{\text{max}}}$$

5. Grouping for Implementation:
$$\text{LogSoftmax}(x_i) = (x_i - z_{\text{max}}) - \log \sum e^{x_j - z_{\text{max}}}$$


#### Core Principle
$$e^{a+b} = e^a \cdot e^b$$


#### Results
* **Without the trick**: The computer tries to calculate `e^1000`, hits the hardware limit, and turns it into inf. Once you have an inf, the rest of the math is destroyed.

* **With the trick**: By "shifting" the values so the maximum is 0, you ensure the computer only ever has to calculate `e^0` or `e^−negative_number`. These are always between 0 and 1, which are the "safest" numbers for a computer to handle.
    - Remember:  `e^−∞ = 0`

### c. Implementation

In [ ]:
import torch


def stable_log_softmax(x):
    # x shape: [Batch, Classes]
    # 1. Find the max value in each row for stability
    # keepdim=True ensures we can subtract it from the original tensor
    z_max = torch.max(x, dim=1, keepdim=True)[0]

    # 2. Subtract the max (Log-Sum-Exp trick)
    # This prevents e^x from becoming infinity.
    # log(sum(exp(x - max))) + max is mathematically equivalent to log(sum(exp(x)))
    log_sum_exp = torch.log(torch.sum(torch.exp(x - z_max), dim=1, keepdim=True))

    return (x - z_max) - log_sum_exp

    # Why this works:
    # The largest value in (x - z_max) is now 0.
    # e^0 is 1, which is perfectly safe for a computer to handle.

/Users/jae.r/projects/ml-study/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


### PyTorch Built-in Alternative

```python
import torch.nn.functional as F

# x shape: [Batch, Classes]
# You MUST specify 'dim' (dimension) to normalize across.
# dim=1 means we calculate the log_softmax across the 'Classes' for each 'Batch'
log_probs = F.log_softmax(x, dim=1)
```

In [3]:
import torch.nn.functional as F
import torch

In [ ]:
def stable_log_softmax(x):
    z_max = torch.max(x, dim=1, keepdim=True)[0]
    log_sum_exp = torch.log(torch.sum(torch.exp(x - z_max), dim=1, keepdim=True))
    return (x - z_max) - log_sum_exp

In [31]:
tensor = torch.tensor([[0.0, 1000.0, 50.0]])
tensor.size()

torch.Size([1, 3])

In [32]:
custom_output = stable_log_softmax(tensor)
official_output = F.log_softmax(tensor, dim=1)

In [33]:
custom_output.size()

torch.Size([1, 3])

In [34]:
official_output.size()

torch.Size([1, 3])

In [35]:
torch.allclose(custom_output, official_output)

True

In [36]:
custom_output

tensor([[-1000.,     0.,  -950.]])

In [37]:
official_output

tensor([[-1000.,     0.,  -950.]])